# Goal

Тестируем стерео-зрение. Это когда один обзервейшн исследуется двумя CNN (178х178->6x6, 72 токена), но с разными смещениями по оси Х. Так же смотрим что лучше работает: когда одна CNN для лева и права (сильная регуляризация), или когда разные (выше гибкость).

# GRID_SEARCH_SPACE

In [ ]:
# @launchit.collect
GRID_SEARCH_SPACE = dict(
    offset=[1, 2, 3, 4, 5],
    is_shared=[True, False],
)

# set_hyperparameters

In [1]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.system.comment = None
    HP.system.random_seed = random.randint(0, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    HP.system.use_amp = True
    
    HP.dataset.train = [
        'train_dataset_seq_eq_4:300',
        'train_dataset_seq_eq_4:301',
        'train_dataset_seq_eq_4:302',
        'train_dataset_seq_eq_4:303',
        'train_dataset_seq_eq_4:304',
        'train_dataset_seq_eq_4:305',
        'train_dataset_seq_eq_4:306',
        'train_dataset_seq_eq_4:307',
        'train_dataset_seq_eq_4:308',
        'train_dataset_seq_eq_4:309',
    ]
    HP.dataset.test = 'test_dataset_seq_eq_4:1'
    offset = optuna_trial.suggest_categorical('offset', GRID_SEARCH_SPACE['offset'])
    HP.dataset.pov_coord_lists = [[[-offset * (2 / 178), 0], [+offset * (2 / 178), 0]]] 
    
    HP.model.parent = None
    HP.model.sequence_length = 4
    HP.model.ob_shape = (1, 178, 152)
    HP.model.ob_vision_head = None
    HP.model.pov_size = 178
    HP.model.povs_count = 2
    
    is_shared = optuna_trial.suggest_categorical('is_shared', GRID_SEARCH_SPACE['is_shared'])
    HP.model.pov_vision_head = dict(grid=(6,6), features_counts=(16, 32, 64, 128), is_shared=is_shared)

    if is_shared:
        HP.model.pov_coords_encoding = dict(frequencies_count=6, inner_d_model=64, is_combined=True)
    else:
        HP.model.pov_coords_encoding = None
        
    HP.model.actions_count = 6
    HP.model.d_model = 256
    HP.model.layers_count = 3
    HP.model.heads_count = 4
    HP.model.render_heads_count = 4
    HP.model.attention_backend = 'EFFICIENT_ATTENTION'
    
    HP.train.epochs_count = 100
    HP.train.batch_size = 128
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = 'const(0.0005)'
    HP.train.bce_loss_coef = 'const(1.0)'
    HP.train.edge_loss_coef = 'const(1.0)'
    
    return HP
# @launchit.stop

# Results
<TBD>

Разбор по итоговым цифрам небольшой. Лучше всего отработал вариант 18c_world_model_02/opt_01/58: offset=4, is_shared=True.

<img src="./img/ssim.png">

Сравнение с вариантом 18c_world_model_02/27 (это когда только один ob_vision_head 178x152->6x6, POV-ов нет, 36 токенов). Как видно выиграш не особо велик, если вообще есть.

<img src="./img/ssim-36-vs-72.png">

**Выводы**
1) смысл в stereo зрении есть
2) вопрос цены: 36 против 72 токенов
3) можно смело начинать в 36 токенов, а если не хватает, уже думать в сторону 72. Но, кажется, для Атари Фростбайт хватит и 36. При это лучше увеличить глубину обзервейшнов